In [ ]:
#| default_exp portfolio

# Portfolio

In [ ]:
#| export
from collections import defaultdict

In [ ]:
#| export
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#| export
from fastcore.basics import *

In [ ]:
#| export
import pandas as pd

In [ ]:
from portfolio.sample_data import *

In [ ]:
#pd.set_option('mode.copy_on_write', True)

Helper function to allow us to slice date-time indices

In [ ]:
#| export
def _slice_dates(data, start, end):
    if not start: start = data.index.min()
    if not end: end = data.index.max()
    return data.loc[start:end]

## A portfolio

To create a portfolio you simply pass the return streams and weights to the portfolio class. It expects monthly return streams. The best way is to gather all your raw, monthly return streams in a single dataframe, including the risk-free (cash-rate) asset. That way you can check for gaps in the data and validate that the data lines up properly before passing it to the portfolio.

The risk free rate will be used to transform the return streams to excess return streams (in excess of risk free).

In [ ]:
#| export
class Portfolio:
    def __init__(self, name:str, return_streams:pd.DataFrame, weights:dict, rf:pd.DataFrame=None, cpi:pd.DataFrame=None, funding_spread:float=0.0, start=None, end=None):
        self.name = name
        self.weights = weights
        self.funding_spread = funding_spread
        assets = list(weights.keys())
        self.full_rf = rf.copy() if rf is not None else None
        self.full_cpi = cpi.copy() if cpi is not None else None
        self.full_rets = return_streams[assets].copy()
        self.rf = _slice_dates(self.full_rf, start, end) if self.full_rf is not None else None
        self.cpi = _slice_dates(self.full_cpi, start, end) if self.full_cpi is not None else None
        self.raw_rets = _slice_dates(self.full_rets, start, end)
        self.return_streams = self.raw_rets.sub(self.rf, axis=0) if self.rf is not None else self.raw_rets.copy()
    def __repr__(self): return str(self.weights)

Let's generate sample data for use in analysis

In [ ]:
rets, rf, cpi = sample_data_se()

In [ ]:
rets.head(2)

In [ ]:
rf.head(2)

In [ ]:
cpi.head(2)

We make a simple 60/40 portfolio

In [ ]:
p = Portfolio('60/40', rets, {'bonds': .40, 'stocks': .60}, rf=rf, cpi=cpi)
p

In [ ]:
p.name

We want to be able to examine performance in different time periods. So it should be easy to re-create portfolio based on specific dates.

In [ ]:
#| export
@patch
def between(self:Portfolio, start=None, end=None):
    "New `Portfolio` sliced to `start`/`end` from the full (never-sliced) history"
    return Portfolio(self.name, self.full_rets, self.weights, rf=self.full_rf, cpi=self.full_cpi, funding_spread=self.funding_spread, start=start, end=end)


In [ ]:
p.return_streams.index.min(), p.return_streams.index.max()

If we want to limit our analysis to between 2010 and 2020 we do like this

In [ ]:
p = p.between('1/1995', '1/2000')

In [ ]:
p.return_streams.index.min(), p.return_streams.index.max()

It is possible to re-slice again to a larger window

In [ ]:
p = p.between('1/1900', '1/2024')

In [ ]:
p.return_streams.index.min(), p.return_streams.index.max()

The portfolio return is the excess return your portfolio makes each month

In [ ]:
#| export
@patch()
def _port_rets(self:Portfolio, excess=True, agg=True):
    stream = self.return_streams if excess else self.raw_rets
    r = (stream*self.weights)
    if agg: r = r.sum(axis=1)
    r.name = self.name
    return r

In [ ]:
#| export
@patch(as_prop=True)
def port_rets(self:Portfolio):
    return self._port_rets()

In [ ]:
p.port_rets.head(2)

## basic portfolio stats

Expected return, vol and sharpe ratio

In [ ]:
#| export
def sharpe(rets: pd.DataFrame):
    'Calculates annualized exp-excess return, vol and sharpe, assumes monthly returns data'
    exp_r = rets.mean() * 12
    vol = rets.std() * 12**0.5
    sharpe = exp_r/vol
    return exp_r,vol,sharpe

In [ ]:
#| export
@patch
def stats(self:Portfolio):
    'Basic portfolio (annualised) stats: exp-excess return, vol and sharpe ratio'
    e,v,s = sharpe(self.port_rets)
    stats = {'expected_return': e, 'volatility': v, 'sharpe_ratio': s}
    return pd.Series(stats, name=self.name)

In [ ]:
s = p.stats()
s

## Portfolio Metrics

Total Compounded Return

shows the portfolio’s total compounded return (not the return in excess of cash).

In [ ]:
#| export
@patch
def cum_return(self:Portfolio):
    'Cumulative compounded total return'
    r = self._port_rets(excess=False)
    return (1+r).cumprod()

In [ ]:
p.cum_return().head(2)

Real Return

In [ ]:
#| export
@patch
def real_r(self: Portfolio, components=False):
    if self.cpi is None:
        raise ValueError("You must specify CPI data to compute real returns")
    r = self._port_rets(excess=False)
    if components: r = self.raw_rets.join(r)
    real_r = (1+r).div(1+self.cpi, axis=0)-1
    return real_r.dropna()

Real Total Wealth. Your wealth less inflation

In [ ]:
#| export
@patch
def real_w(self:Portfolio):
    'Total real wealth'
    r = self.real_r()
    real_w = (1+r).cumprod()
    real_w.name = self.name
    return real_w

In [ ]:
p.real_w().head(2)

Rolling Return. Metric is computed in annualised terms. For a 5 year rolling window return, each point is a return over 5 years expressed as an annualised return

In [ ]:
#| export
@patch
def roll_return(self:Portfolio, months=12, excess=True, extras=False, forward=False):
    'Rolling (excess) return (linearly summed) expressed in annualised terms.'
    years = months/12
    r = self._port_rets(excess=excess)
    if extras: r = r.to_frame().join(self.rf).join(self.cpi).dropna()
    if forward: r = r[::-1]
    r = r.rolling(months).sum().dropna()
    if forward: r = r[::-1]
    return r/years

The default is excess returns

In [ ]:
p.roll_return(24).plot()

We can also plot cpi and rf to get get a better feel of returns and conditions

In [ ]:
p.roll_return(24, excess=False, extras=True).plot()

Cumulative Excess Return. Shows how portfolio performs each month, upwards sloping means we are doing better than risk-free, downward sloping means we are doing worse. Flat means we are same as excess. Does not use compounding.

In [ ]:
#| export
@patch
def cum_excess_return(self:Portfolio):
    "Cumulative sum of monthly returns (no compounding)"
    return self.port_rets.cumsum()

In [ ]:
p.cum_excess_return().plot()

Detrended Cumulative Return

By subtracting the long-run average monthly return (μ) before cumulating, we remove the expected drift and are left with only the unexpected component — the wiggles. This makes it much easier to see environmental biases: periods where an asset consistently outperformed or underperformed its own long-run average.

In [ ]:
#| export
@patch
def detrended_cum_return(self:Portfolio):
    "Cumulative sum of returns minus long-run mean — removes drift, shows environmental cycles"
    r = self.port_rets
    return (r - r.mean()).cumsum()

In [ ]:
p.detrended_cum_return().plot()

## Comparing portfolios

In [ ]:
p1 = Portfolio("60/40", rets, weights={'stocks': 0.60, 'bonds': 0.40}, rf=rf)
p2 = Portfolio("Risk Parity", rets, weights={'stocks': 0.42, 'bonds': 0.58}, rf=rf)

If we have two or more portfolios we should be able to easily compare metrics

In [ ]:
#| export
def compare(*portfolios, metric='stats', **kwargs):
    df = pd.concat([getattr(p, metric)(**kwargs) for p in portfolios], axis=1)
    df.name = metric
    return df

In [ ]:
compare(p1,p2)

We may choose any of the above defined metrics or methods.

In [ ]:
compare(p1,p2, metric='cum_return').tail(2)

Arguments to metrics are passed in last as keyword args.

In [ ]:
compare(p1,p2, metric='roll_return', months=24).tail(2)

## Portfolio inspection (asset level contribution)

Statistics for each asset that makes up the portfolio. These shows the actual weighted returns as given by portfolio construction.

In [ ]:
#| export
@patch(as_prop=True)
def asset_rets(self:Portfolio):
    return (self.return_streams*self.weights)[self.weights.keys()]

In [ ]:
#| export
@patch
def asset_stats(self:Portfolio):
    'Basic portfolio (annualised) stats: exp return, vol and sharpe ratio'
    e,v,s = sharpe(self.asset_rets)
    stats = {'expected_return': e, 'volatility': v, 'sharpe_ratio': s}
    return pd.DataFrame(stats).T

In [ ]:
p.asset_stats()

Risk Contribution

Risk contribution breaks portfolio volatility into per-asset components. Each asset's contribution is accounting for both weight and how much each asset co-moves with the total portfolio. These sum exactly to σ_p, unlike a simple weighted sum of individual vols which ignores diversification.

If a large percentage comes from a single asset it means that this asset is dominating the volatiltiy and hence risk/return profile of the portfolio.

In [ ]:
#| export
@patch
def risk_contribution(self:Portfolio):
    "Per-asset risk contribution as % of portfolio variance (sums to 1)"
    cov = self.asset_rets.join(self.port_rets).cov() * 12
    pv = cov.loc[self.name, self.name]
    rc = {a: cov.loc[a, self.name] / pv for a in self.weights}
    return pd.Series(rc, name=self.name)

In [ ]:
p.risk_contribution()

Correlation

In [ ]:
#| export
@patch
def correlation(self:Portfolio):
    "Pairwise correlation matrix of underlying assets and portfolio"
    return self.asset_rets.join(self.port_rets).corr()

In [ ]:
p.correlation()

Driver of portfolio returns and volatility plot

In [ ]:
#| export
@patch
def return_drivers(self:Portfolio, months=12*5):
    "Calculates portfolio and asset rolling return mean centered"
    return self.asset_rets.join(self.port_rets).rolling(months).agg(sum).dropna()

## Drawdowns

A drawdown at time t is the percentage decline from the portfolio's previous peak to its current value. Max drawdown is the worst peak-to-trough decline over the full history — a key measure of downside risk that Sharpe ratio alone misses entirely.

In [ ]:
#| export
@patch
def drawdown_series(self:Portfolio, assets=False):
    "Time series of drawdowns from rolling peak"
    wealth = self.cum_return()
    if assets: wealth = (1+self._port_rets(excess=False, agg=False)).cumprod().join(wealth)
    return wealth / wealth.cummax() - 1

In [ ]:
p.drawdown_series().plot()

In [ ]:
#| export
@patch
def max_drawdown(self:Portfolio):
    "Worst peak-to-trough decline over full history"
    return self.drawdown_series().min().item()

In [ ]:
p.max_drawdown()

## Finding weak periods

Real Rolling return per decade (ann). Makes it easy to identify periods where real returns were low

In [ ]:
#| export
@patch
def lost_decade(self: Portfolio, components=False, months=120):
    "Real forward 10y returns (ann)"
    real_r = self.real_r(components=components)
    annual = (1 + real_r)[::-1].rolling(months).agg(np.prod)[::-1] ** (1/10) - 1
    return annual.dropna()

In [ ]:
p.lost_decade(components=True).plot()

Decade real return distribution. Allow us to visualize how total return has varied by decade.

In [ ]:
#| export
@patch
def r_by_decade(self: Portfolio):
    "Real return series per decade"
    decade_starts = [y for y in range(1900, 2030, 10)]
    blocks = [self.real_r().loc[f'{y}-01-01':f'{y+9}-12-31'] for y in decade_starts]
    blocks = [b for b in blocks if len(b) == 120]
    df = [pd.concat([pd.Series([1]), 1+b]).cumprod().reset_index(drop=True) for b in blocks]
    df = pd.concat(df, axis=1)
    df.columns = [f'{b.index[0].year}s' for b in blocks]
    df.index = df.index/12 # normalize months to years
    return df

In [ ]:
p.r_by_decade().plot(logy=True)

## Leverage

Creating a leveraged portfolio is easy. Simply specify weights that add up to more than 100%

In [ ]:
p_l = Portfolio("Leveraged", rets, weights={'stocks': 0.65, 'bonds': 0.65}, rf=rf)
p_l

In [ ]:
p_u = Portfolio("Unlevered", rets, weights={'stocks': 0.50, 'bonds': 0.50}, rf=rf)
p_u

Leveraging increases the volatility and return linearly. We are moving out on the risk curve.

In [ ]:
compare(p_l, p_u)

As a result we also get larger swings and drawdowns

In [ ]:
compare(p_u, p_l, metric='drawdown_series').plot()

In [ ]:
compare(p_u, p_l, metric='cum_return').plot()

## Risk Parity Weighing

In [ ]:
import numpy as np
from scipy.optimize import minimize

In [ ]:
def risk_parity_weights(rets, risk_shares):
    "Compute weights given asset returns df and dict of desired risk shares"
    assets = list(risk_shares.keys())
    C = rets[assets].cov().values * 12
    r = np.array([risk_shares[a] for a in assets])
    n = len(assets)
    def obj(w): return 0.5*(w@C@w) - sum(r[i]*np.log(w[i]) for i in range(n))
    def grad(w): return C@w - r/w
    w0 = np.ones(n)/n
    bounds = [(1e-6, None)]*n
    res = minimize(obj, w0, jac=grad, method='L-BFGS-B', bounds=bounds)
    w = res.x / res.x.sum()
    return dict(zip(assets, w))

In [ ]:
risk_shares = dict(bonds=0.50, stocks=0.50)
w = risk_parity_weights(p.return_streams, risk_shares)
allw = Portfolio('allw', p.return_streams, w)
allw.risk_contribution()

## Risk

Annualised per-asset vols computed over a trailing window. Useful on its own to see how each asset's risk has shifted through time.

In [ ]:
#| export
@patch
def rolling_vol(self:Portfolio, months=24):
    "Annualised rolling vol per asset over `months` window"
    return self._port_rets(excess=False, agg=False).rolling(months).agg(lambda x: x.std() * 12**0.5).dropna()

In [ ]:
p.rolling_vol().plot()

We can normalize these volatilities to get a simple measure of risk contribution. This ignores covariance and treats each asset as independent, assuming portfolio vol is simply sum of individual asset vols.

In [ ]:
#| export
@patch
def rc_simple(self:Portfolio):
    vol = self.rolling_vol()
    tot = vol.sum(axis=1)
    rc = vol.div(tot, axis=0)
    return rc

In [ ]:
rc = p.rc_simple()

In [ ]:
rc.plot()

## Plotting Utils

Making dataframe and data series plots a bit nicer

In [ ]:
#| export
def _plot(data, log, title):
    m = np.log2(data) if log else data
    ax = m.plot(figsize=(12,6), title=title)
    if log:
        ymin,ymax = int(np.floor(m.min().min())), int(np.ceil(m.max().max()))
        ax.set_yticks(range(ymin, ymax+1))
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{2.**y:.1f}x'))
    ax.grid(True, which='both', alpha=0.3)
    return ax

In [ ]:
#| export
@patch
def portfolio_plot(self:pd.DataFrame, log=False):
    title = getattr(self, 'name', ', '.join(self.columns))
    return _plot(self, log, title)

In [ ]:
#| export
@patch
def portfolio_plot(self:pd.Series, log=False):
    title = getattr(self, 'name', 'unknown')
    return _plot(self, log, title)

In [ ]:
compare(p_u, p_l, metric='cum_return').portfolio_plot(log=True)